# 00. Environment Setup and BigQuery Initialization

## 0. Overview
This notebook initializes the analysis environment and verifies connectivity to the MIMIC-IV BigQuery datasets. It serves as the entry point for all downstream cohort construction and survival analysis notebooks (01–04).

No analytic transformations are performed in this notebook.

## 1. Purpose
The purpose of this notebook is to ensure a reproducible and consistent execution environment, including Python dependencies, virtual environment configuration, and BigQuery authentication.

## 2. Data Sources
- **MIMIC-IV v3.1** (BigQuery public dataset)
- Project: `mimic-iv-portfolio`
- Working dataset: `nonicu_raas`

## 3. Environment Setup
- Python virtual environment activation
- Package version checks
- Jupyter kernel verification

In [ ]:
# Confirm the exact Python interpreter path.
import sys

print(sys.executable)


In [ ]:
# Core imports and project configuration.
from google.api_core.exceptions import NotFound
from google.auth import default
from google.cloud import bigquery

PROJECT_ID = "mimic-iv-portfolio"
DATASET_ID = "nonicu_raas"
LOCATION = "US"

# Lowercase aliases are kept for consistency with downstream notebooks.
project_id = PROJECT_ID
dataset_id = DATASET_ID



## 4. BigQuery Initialization
- Authentication and client initialization
- Verification of BigQuery access
- Minimal test query against MIMIC-IV tables

In [ ]:
# Authenticate with Application Default Credentials (ADC) and initialize BigQuery once.
# Run `gcloud auth application-default login` outside this notebook if ADC is not configured.
credentials, adc_project_id = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
client = bigquery.Client(credentials=credentials, project=project_id)

print("Credentials type:", type(credentials).__name__)
print("ADC project:", adc_project_id)
print("BigQuery client project:", client.project)


def dataset_reference(dataset_name: str = dataset_id) -> bigquery.Dataset:
    """Return a BigQuery Dataset object for the configured project and dataset."""
    dataset = bigquery.Dataset(f"{project_id}.{dataset_name}")
    dataset.location = LOCATION
    return dataset


def ensure_dataset_exists(dataset_name: str = dataset_id) -> bigquery.Dataset:
    """Create the working dataset if it does not already exist."""
    dataset = dataset_reference(dataset_name)
    try:
        return client.get_dataset(dataset)
    except NotFound:
        return client.create_dataset(dataset)


In [ ]:
# Ensure the working dataset exists in the configured location.
dataset = ensure_dataset_exists(dataset_id)
print(f"Dataset `{dataset.full_dataset_id}` is available in {dataset.location}.")

## 5. Sanity Checks
- Confirmation of successful BigQuery connection
- Validation of dataset and table visibility

In [ ]:
query = """
SELECT COUNT(*) AS icu_stays
FROM `physionet-data.mimiciv_3_1_icu.icustays` # Extract Base ICU Stays (mimiciv_icu.icustays)
"""
df = client.query(query).to_dataframe()
print(df)

## 6. Outputs and Downstream Use
This notebook produces no analytic dataset. 
 
It ensures that all subsequent notebooks (01–04) can be executed reproducibly
without environment-related errors.